In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [3]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [4]:
# connect to localhost
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  database='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [5]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

---
공시지가 public_land_price

In [ ]:
# 공시지가 - m1.land_price
job = client.query(
  f'''
  select
    pnu,
    base_year,
    amount
  from m1.land_price
  where
    left(pnu,2) = '11' and
    base_year >= '2020'
  '''
)
public_land_price_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
try:
  public_land_price_df.to_sql(
    'public_land_price',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

지하철 역 출입구 subway_ent

In [ ]:
job = client.query(
  f'''
  select
    station_nm,
    line_nm,
    ent_num ent_nm,
    lat lon,
    lon lat
  from m1.subway_mst_xy
  where region_gb = '수도권'
  '''
)
subway_ent_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [19]:
subway_ent_df['geom'] = 'POINT (' + subway_ent_df['lon'].astype('string') + ' ' + subway_ent_df['lat'].astype('string') + ')'

In [20]:
try:
  cursor.execute(
    f'''
    create table subway_ent (
      station_nm varchar,
      line_nm varchar,
      ent_nm varchar,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [23]:
try:
  cursor.execute(
    'delete from subway_ent'
  )
  subway_ent_df[['station_nm','line_nm','ent_nm','geom']].to_sql(
    'subway_ent',
    engine,
    if_exists='append',
    index=False,
  )
except Exception as err:
  print(err)

In [19]:
try:
  cursor.execute(
    'create index idx_subway_ent_geom on subway_ent using gist(geom)'
  )
except Exception as err:
  print(err)

건축물대장 표제부 building_info

In [ ]:
job = client.query(
  f'''
	SELECT
		concat(
			sigungu_cd,
			bjdong_cd,
			case
				when plat_gb_cd = '0' then '1'
				when plat_gb_cd = '1' then '2'
				else '0'
			end,
			bun,
			ji
		) pnu,
		mgm_bldrgst_pk,
    main_purps_cd_nm use_nm,
		plat_area,
		arch_area,
		tot_area,
		ride_use_elvt_cnt + emgen_use_elvt_cnt elev_cnt,
		indr_mech_utcnt + oudr_mech_utcnt + indr_auto_utcnt + oudr_auto_utcnt parklot_cnt,
		useapr_day complete_dt
	from m1.bld_rgst
	where
	base_dt = '2025-04-01' and
	left(sigungu_cd,2) = '11' and
	main_atch_gb_cd = '0'
	'''
)
bld_rgst_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [7]:
try:
  bld_rgst_df.to_sql(
    'building_info',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

유동인구 walk_pop

In [ ]:
job = client.query(
	f'''
	select
		base_dt,
		cell_id,
		ctdo_cd,
		emd_cd,
		adng_cd,
		corc_walk_popl_cnt tot_cnt,
		round(corc_walk_popl_cnt * walk_male_pt/100,2) male_cnt,
		round(corc_walk_popl_cnt * (100-walk_male_pt)/100,2) female_cnt,
		round(corc_walk_popl_cnt * walk_u20_pt/100,2) age_u20_cnt,
		round(corc_walk_popl_cnt * walk_20_pt/100,2) age_20_cnt,
		round(corc_walk_popl_cnt * walk_30_pt/100,2) age_30_cnt,
		round(corc_walk_popl_cnt * walk_40_pt/100,2) age_40_cnt,
		round(corc_walk_popl_cnt * walk_50_pt/100,2) age_50_cnt,
		round(corc_walk_popl_cnt * (100-walk_u20_pt-walk_20_pt-walk_30_pt-walk_40_pt-walk_50_pt),2) age_o50_cnt,
		round(corc_walk_popl_cnt * tmzn_0810_pt/100,2) time_0810_cnt,
		round(corc_walk_popl_cnt * tmzn_1113_pt/100,2) time_1113_cnt,
		round(corc_walk_popl_cnt * tmzn_1416_pt/100,2) time_1416_cnt,
		round(corc_walk_popl_cnt * tmzn_1719_pt/100,2) time_1719_cnt,
		round(corc_walk_popl_cnt * tmzn_2022_pt/100,2) time_2022_cnt,
		round(corc_walk_popl_cnt * (100-tmzn_0810_pt-tmzn_1113_pt-tmzn_1416_pt-tmzn_1719_pt-tmzn_2022_pt)/100,2) time_2307_cnt,
		st_astext(point) geom
	from m1.walk_pop
	where ctdo_cd = '11'
	'''
)
walk_pop_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
try:
  cursor.execute(
    f'''
    create table walk_pop (
      base_dt date,
      cell_id varchar,
      ctdo_cd varchar,
      emd_cd varchar,
      adng_cd varchar,
      tot_cnt numeric,
      male_cnt numeric,
      female_cnt numeric,
      age_u20_cnt numeric,
      age_20_cnt numeric,
      age_30_cnt numeric,
      age_40_cnt numeric,
      age_50_cnt numeric,
      age_o50_cnt numeric,
      time_0810_cnt numeric,
      time_1113_cnt numeric,
      time_1416_cnt numeric,
      time_1719_cnt numeric,
      time_2022_cnt numeric,
      time_2307_cnt numeric,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [12]:
try:
  cursor.execute(
    'delete from walk_pop'
  )
  walk_pop_df.to_sql(
    'walk_pop',
    engine,
    if_exists='append',
    index=False,
  )
except Exception as err:
  print(err)

In [20]:
try:
  cursor.execute(
    'create index idx_walk_pop_geom on walk_pop using gist(geom)'
  )
except Exception as err:
  print(err)

지역별 임대료, 분기별 region_rent

In [ ]:
job = client.query(
  f'''
  select
    base_dt,
    rent_fee,
    buld_gbn,
    level_no_out,
    metro_nm,
    region_cd,
    region_nm,
    research_date,
    section_nm,
    sido_nm
  from m1.lease_price
  where
    sido_nm = '서울' and
    buld_gbn = '3' and
    base_dt >= '2020-01-01' and
    base_dt < '2025-03-01'
  '''
)
region_rent_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [15]:
try:
  region_rent_df.to_sql(
    'region_rent',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

지역별 매출액 + 특성별 sig_sales, emd_sales

In [15]:
job = client.query(
  f'''
  select
    base_dt,
    sig_cd,
    count(store_no) store_cnt,
    sum(est_amt) sales_amt,
    sum(est_cnt) sales_cnt,
    sum(delivery_amount) delivery_amt,
    sum(delivery_count) delivery_cnt,
    round(sum(
      case
        when category_first_name = '외식' then est_amt
        else 0
      end
    )) sales_restaurant_amt,
    round(sum(
      case
        when category_first_name = '서비스' then est_amt
        else 0
      end
    )) sales_service_amt,
    round(sum(
      case
        when category_first_name = '도소매' then est_amt
        else 0
      end
    )) sales_retail_amt,
    round(sum(
      case
        when category_first_name = '기타' then est_amt
        else 0
      end
    )) sales_etc_amt,
    round(sum(est_amt*wk_rt/100)) sales_wk_amt,
    round(sum(est_amt*we_rt/100)) sales_we_amt,
    round(sum(est_amt*time_0510_rt/100)) sales_morning_amt,
    round(sum(est_amt*(time_1114_rt+time_1517_rt)/100)) sales_afternoon_amt,
    round(sum(est_amt*(time_1819_rt+time_2021_rt)/100)) sales_evening_amt,
    round(sum(est_amt*(time_2224_rt+time_0104_rt)/100)) sales_night_amt,
    round(sum(est_amt*(m20_rt+m30_rt+m40_rt+m50_rt+m60_rt)/100)) sales_male_amt,
    round(sum(est_amt*(f20_rt+f30_rt+f40_rt+f50_rt+f60_rt)/100)) sales_female_amt,
    round(sum(est_amt*(f20_rt+m20_rt)/100)) sales_age20_amt,
    round(sum(est_amt*(f30_rt+m30_rt)/100)) sales_age30_amt,
    round(sum(est_amt*(f40_rt+m40_rt)/100)) sales_age40_amt,
    round(sum(est_amt*(f50_rt+m50_rt)/100)) sales_age50_amt,
    round(sum(est_amt*(f60_rt+m60_rt)/100)) sales_age60_amt
  from m2.sh_bldg_sales
  where left(pnu,2) = '11'
  group by 1,2
  '''
)
sig_sales = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
try:
  sig_sales.to_sql(
    'sig_sales',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

In [13]:
job = client.query(
  f'''
  select
    base_dt,
    emd_cd,
    count(store_no) store_cnt,
    sum(est_amt) sales_amt,
    sum(est_cnt) sales_cnt,
    sum(delivery_amount) delivery_amt,
    sum(delivery_count) delivery_cnt,
    round(sum(
      case
        when category_first_name = '외식' then est_amt
        else 0
      end
    )) sales_restaurant_amt,
    round(sum(
      case
        when category_first_name = '서비스' then est_amt
        else 0
      end
    )) sales_service_amt,
    round(sum(
      case
        when category_first_name = '도소매' then est_amt
        else 0
      end
    )) sales_retail_amt,
    round(sum(
      case
        when category_first_name = '기타' then est_amt
        else 0
      end
    )) sales_etc_amt,
    round(sum(est_amt*wk_rt/100)) sales_wk_amt,
    round(sum(est_amt*we_rt/100)) sales_we_amt,
    round(sum(est_amt*time_0510_rt/100)) sales_morning_amt,
    round(sum(est_amt*(time_1114_rt+time_1517_rt)/100)) sales_afternoon_amt,
    round(sum(est_amt*(time_1819_rt+time_2021_rt)/100)) sales_evening_amt,
    round(sum(est_amt*(time_2224_rt+time_0104_rt)/100)) sales_night_amt,
    round(sum(est_amt*(m20_rt+m30_rt+m40_rt+m50_rt+m60_rt)/100)) sales_male_amt,
    round(sum(est_amt*(f20_rt+f30_rt+f40_rt+f50_rt+f60_rt)/100)) sales_female_amt,
    round(sum(est_amt*(f20_rt+m20_rt)/100)) sales_age20_amt,
    round(sum(est_amt*(f30_rt+m30_rt)/100)) sales_age30_amt,
    round(sum(est_amt*(f40_rt+m40_rt)/100)) sales_age40_amt,
    round(sum(est_amt*(f50_rt+m50_rt)/100)) sales_age50_amt,
    round(sum(est_amt*(f60_rt+m60_rt)/100)) sales_age60_amt
  from m2.sh_bldg_sales
  where left(pnu,2) = '11'
  group by 1,2
  '''
)
emd_sales = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [14]:
try:
  emd_sales.to_sql(
    'emd_sales',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

상권별 매출액 + 특성별 market_sales

In [16]:
job = client.query(
  f'''
  select
    store.base_dt,
    market.site_name market_name,
    market.site_area market_area,
    count(store_no) store_cnt,
    sum(est_amt) sales_amt,
    sum(est_cnt) sales_cnt,
    sum(delivery_amount) delivery_amt,
    sum(delivery_count) delivery_cnt,
    round(sum(
      case
        when category_first_name = '외식' then est_amt
        else 0
      end
    )) sales_restaurant_amt,
    round(sum(
      case
        when category_first_name = '서비스' then est_amt
        else 0
      end
    )) sales_service_amt,
    round(sum(
      case
        when category_first_name = '도소매' then est_amt
        else 0
      end
    )) sales_retail_amt,
    round(sum(
      case
        when category_first_name = '기타' then est_amt
        else 0
      end
    )) sales_etc_amt,
    round(sum(est_amt*wk_rt/100)) sales_wk_amt,
    round(sum(est_amt*we_rt/100)) sales_we_amt,
    round(sum(est_amt*time_0510_rt/100)) sales_morning_amt,
    round(sum(est_amt*(time_1114_rt+time_1517_rt)/100)) sales_afternoon_amt,
    round(sum(est_amt*(time_1819_rt+time_2021_rt)/100)) sales_evening_amt,
    round(sum(est_amt*(time_2224_rt+time_0104_rt)/100)) sales_night_amt,
    round(sum(est_amt*(m20_rt+m30_rt+m40_rt+m50_rt+m60_rt)/100)) sales_male_amt,
    round(sum(est_amt*(f20_rt+f30_rt+f40_rt+f50_rt+f60_rt)/100)) sales_female_amt,
    round(sum(est_amt*(f20_rt+m20_rt)/100)) sales_age20_amt,
    round(sum(est_amt*(f30_rt+m30_rt)/100)) sales_age30_amt,
    round(sum(est_amt*(f40_rt+m40_rt)/100)) sales_age40_amt,
    round(sum(est_amt*(f50_rt+m50_rt)/100)) sales_age50_amt,
    round(sum(est_amt*(f60_rt+m60_rt)/100)) sales_age60_amt
  from m2.sh_bldg_sales store,
  (
  select
    site_name,
    site_area,
    geometry geom
  from biz.market_site_2024
  where sd_code = '11'
  ) as market
  where
    st_intersects(
      market.geom,
      store.point
    )
  group by 1,2,3
  '''
)
market_sales = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [18]:
try:
  market_sales.to_sql(
    'market_sales',
    engine,
    if_exists='replace',
    index=False
  )
except Exception as err:
  print(err)

학교 (어린이집/유치원/초등학교/중학교/고등학교/대학교)

In [9]:
job = client.query(
  f'''
  select
    school_nm,
    school_gb_nm school_type,
    st_astext(point) geom
  from m1.school_xy
  '''
)
school_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
try:
  cursor.execute(
    f'''
    create table school (
      school_nm varchar,
      school_type varchar,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [ ]:
cursor.execute(
  'delete from school'
)
school_df.to_sql(
  'school',
  engine,
  if_exists='append',
  index=False
)

412

In [18]:
try:
  cursor.execute(
    f'''
    create index idx_school_geom on school using GIST(geom);
    alter table school add geom_3857 geometry(geometry,3857);
    update school set geom_3857 = st_transform(geom,3857);
    create index idx_school_geom_3857 on school using GIST(geom_3857);
    '''
  )
except Exception as err:
  print(err)

거주인구

In [22]:
job = client.query(
  f'''
  select
    pop.pnu,
    pop.pop_cnt,
    lot.point,
    lot.geom
  from (
    select
      pnu,
      pop_cnt
    from m2.live_pop
    where
      left(pnu,2) = '11' and
      base_dt = '2024-12-01'
  ) pop
  left join (
    select
      pnu,
      st_centroid(poly) point,
      poly geom
    from m1.land_map_new
    where left(pnu,2) = '11'
  ) lot
  on pop.pnu = lot.pnu
  '''
)
live_pop = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [23]:
try:
  cursor.execute(
    f'''
    create table live_pop (
      pnu varchar,
      pop_cnt numeric,
      point geometry(geometry,4326),
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [24]:
cursor.execute(
  'delete from live_pop'
)
live_pop.to_sql(
  'live_pop',
  engine,
  if_exists='append',
  index=False
)

251

In [25]:
try:
  cursor.execute(
    f'''
    create index idx_live_pop_point on live_pop using GIST(point);
    alter table live_pop add point_3857 geometry(geometry,3857);
    update live_pop set point_3857 = st_transform(point,3857);
    create index idx_live_pop_point_3857 on live_pop using GIST(point_3857);
    '''
  )
except Exception as err:
  print(err)

In [26]:
try:
  cursor.execute(
    f'''
    create index idx_live_pop_geom on live_pop using GIST(geom);
    alter table live_pop add geom_3857 geometry(geometry,3857);
    update live_pop set geom_3857 = st_transform(geom,3857);
    create index idx_live_pop_geom_3857 on live_pop using GIST(geom_3857);
    '''
  )
except Exception as err:
  print(err)

직장인구

In [27]:
job = client.query(
  f'''
  select
    pop.pnu,
    pop.pop_cnt,
    lot.point,
    lot.geom
  from (
    select
      pnu,
      pop_cnt
    from m2.work_pop
    where
      left(pnu,2) = '11' and
      base_dt = '2024-12-01'
  ) pop
  left join (
    select
      pnu,
      st_centroid(poly) point,
      poly geom
    from m1.land_map_new
    where left(pnu,2) = '11'
  ) lot
  on pop.pnu = lot.pnu
  '''
)
work_pop = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [29]:
try:
  cursor.execute(
    f'''
    create table work_pop (
      pnu varchar,
      pop_cnt numeric,
      point geometry(geometry,4326),
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [30]:
cursor.execute(
  'delete from work_pop'
)
work_pop.to_sql(
  'work_pop',
  engine,
  if_exists='append',
  index=False
)

651

In [31]:
try:
  cursor.execute(
    f'''
    create index idx_work_pop_point on work_pop using GIST(point);
    alter table work_pop add point_3857 geometry(geometry,3857);
    update work_pop set point_3857 = st_transform(point,3857);
    create index idx_work_pop_point_3857 on work_pop using GIST(point_3857);
    '''
  )
except Exception as err:
  print(err)

In [32]:
try:
  cursor.execute(
    f'''
    create index idx_work_pop_geom on work_pop using GIST(geom);
    alter table work_pop add geom_3857 geometry(geometry,3857);
    update work_pop set geom_3857 = st_transform(geom,3857);
    create index idx_work_pop_geom_3857 on work_pop using GIST(geom_3857);
    '''
  )
except Exception as err:
  print(err)

토지 좌표

In [15]:
job = client.query(
  f'''
  select
    pnu,
    st_astext(st_centroid(poly)) point,
    st_astext(poly) geom
  from m1.land_map_new
  where left(pnu,2) = '11'
  '''
)
lot_polygon = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [17]:
try:
  cursor.execute(
    f'''
    create table lot_polygon (
      pnu varchar(19),
      point geometry(geometry,4326),
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [19]:
try:
  cursor.execute('delete from lot_polygon')
  lot_polygon.to_sql(
    'lot_polygon',
    engine,
    if_exists='append',
    index=False
  )
except Exception as err:
  print(err)

In [20]:
try:
  cursor.execute(
    f'''
    create index idx_lot_polygon_point on lot_polygon using GIST(point);
    alter table lot_polygon add point_3857 geometry(geometry,3857);
    update lot_polygon set point_3857 = st_transform(point,3857);
    create index idx_lot_polygon_point_3857 on lot_polygon using GIST(point_3857);
    '''
  )
except Exception as err:
  print(err)

In [21]:
try:
  cursor.execute(
    f'''
    create index idx_lot_polygon_geom on lot_polygon using GIST(geom);
    alter table lot_polygon add geom_3857 geometry(geometry,3857);
    update lot_polygon set geom_3857 = st_transform(geom,3857);
    create index idx_lot_polygon_geom_3857 on lot_polygon using GIST(geom_3857);
    '''
  )
except Exception as err:
  print(err)